# TF32 Tensor Core GEMM on A100: 직접 짠 CUDA 커널 vs cuBLAS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/tensor_core_gemm_a100.ipynb)

[tensor_core_gemm](../tensor_core_gemm/README.md)의 CUDA C++ 커널 7개를 빌드하고, 같은 바이너리 안에서 **cuBLAS FP32(SIMT)**·**cuBLAS TF32(Tensor Core)**와 비교한다.
입출력은 SGEMM_CUDA와 같은 FP32 row-major `C = α·A·B + β·C`(α=0.5, β=3)이다.

| # | 커널 | 추가되는 개념 |
|---|---|---|
| 0 | cuBLAS `COMPUTE_32F` | SIMT FP32 기준선 (피크 19.5 TFLOPs) |
| 1 | WMMA naive | `nvcuda::wmma` 16×16×8, global에서 직접 load |
| 2 | WMMA + SMEM | 128×128×16 block tile, 64×32 warp tile |
| 3 | `mma.sync` | PTX m16n8k8 TF32, fragment 레이아웃 직접 계산 |
| 4 | `cp.async` | 3-stage SMEM 파이프라인 |
| 5 | swizzle | XOR swizzle, BK=32, 128×256 타일, coalesced epilogue |
| 6 | vectorized frag | k·n 순열로 fragment LDS.128, 정수 덧셈 TF32 반올림, 주소 계산 루프 밖으로 |
| 7 | frag double buffer | fragment 더블 버퍼링, barrier 앞당기기, block rasterization |
| 100 | cuBLAS `COMPUTE_32F_FAST_TF32` | Tensor Core 목표 (피크 156 TFLOPs) |

**런타임 설정**: `런타임 → 런타임 유형 변경 → A100 GPU`. sm_80 미만 GPU에서는 바이너리가 실행을 거부한다.

## 0. 환경 확인과 빌드

In [ ]:
import os, subprocess
from pathlib import Path

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,clocks.max.sm", "--format=csv"],
                     capture_output=True, text=True).stdout)
print(subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout.splitlines()[-1])

# 로컬 체크아웃(notebooks/ 안에서 실행)이면 그대로 쓰고, Colab이면 저장소를 받는다.
if Path("../tensor_core_gemm/Makefile").exists():
    ROOT = Path("../tensor_core_gemm").resolve()
else:
    if not Path("tensor2silicon").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Jake-Song/tensor2silicon.git"], check=True)
    ROOT = Path("tensor2silicon/tensor_core_gemm").resolve()
print("source:", ROOT)

build = subprocess.run(["make", "-C", str(ROOT), "build", "regs"], capture_output=True, text=True)
print(build.stdout[-4000:], build.stderr[-4000:])
assert build.returncode == 0, "빌드 실패"
BIN = ROOT / "build" / "tc_gemm"

## 1. 실행과 파싱

각 커널을 128 ~ 8192 정사각 크기에서 실행한다. 바이너리는 크기마다 다음 순서로 동작한다.

1. 같은 초기 C에서 cuBLAS FP32 결과와 비교한다. 지표는 `max|diff| / max|ref|`이고, 1e-3 미만이면 PASS다.
2. warmup 1회 뒤 50회 반복 시간을 `cudaEvent`로 측정한다.

kernel 100(cuBLAS TF32)의 상대 오차가 TF32 정밀도에서 기대할 수 있는 기준값이다.

In [ ]:
import re
import pandas as pd

KERNELS = {0: "cuBLAS FP32", 100: "cuBLAS TF32", 1: "1 WMMA naive", 2: "2 WMMA+SMEM",
           3: "3 mma.sync", 4: "4 cp.async", 5: "5 swizzle", 6: "6 vec frag", 7: "7 frag pipe"}
MAX_SIZE = 8192

RE_DIM = re.compile(r"dimensions\(m=n=k\) (\d+)")
RE_ERR = re.compile(r"rel ([0-9.eE+-]+) -> (PASS|FAIL)")
RE_PERF = re.compile(r"performance: \(\s*([0-9.]+)\) GFLOPS\. size: \((\d+)\)")

rows, logs = [], {}
for k, name in KERNELS.items():
    out = subprocess.run([str(BIN), str(k), str(MAX_SIZE)], capture_output=True, text=True)
    logs[k] = out.stdout + out.stderr
    size, err = None, {}
    for line in logs[k].splitlines():
        if m := RE_DIM.search(line):
            size = int(m.group(1))
        elif m := RE_ERR.search(line):
            err[size] = (float(m.group(1)), m.group(2))
        elif m := RE_PERF.search(line):
            s = int(m.group(2))
            rel, status = err.get(s, (float("nan"), "ref"))
            rows.append(dict(kernel=name, size=s, tflops=float(m.group(1)) / 1e3,
                             rel_err=rel, status=status))
    print(f"{name:<14} exit={out.returncode}")
    if out.returncode != 0:
        print(logs[k][-2000:])

df = pd.DataFrame(rows)
order = list(KERNELS.values())

## 2. 정확도: cuBLAS FP32 대비 상대 오차

In [ ]:
acc = df[df.kernel != "cuBLAS FP32"].pivot(index="kernel", columns="size", values="rel_err")
display(acc.reindex([o for o in order if o in acc.index]).map(lambda x: f"{x:.2e}"))
fails = df[df.status == "FAIL"]
print("FAIL 없음" if fails.empty else fails)

## 3. 성능: cuBLAS 대비

- `TFLOPs/s` = 2·N³ / 평균 시간
- `% cuBLAS TF32`: 같은 Tensor Core 경로를 쓰는 cuBLAS 대비 비율 (주 비교 대상)
- `× cuBLAS FP32`: SIMT FP32 cuBLAS 대비 배수 (SGEMM_CUDA가 넘어설 수 없던 기준)

In [ ]:
tf = df.pivot(index="kernel", columns="size", values="tflops").reindex(order)
display(tf.round(1))

ref_tf32 = tf.loc["cuBLAS TF32"]
ref_fp32 = tf.loc["cuBLAS FP32"]
display((100 * tf / ref_tf32).round(1).rename_axis("% of cuBLAS TF32"))
display((tf / ref_fp32).round(2).rename_axis("x cuBLAS FP32"))

In [ ]:
PEAK_TF32 = 156.0
summary = []
for s in (4096, 8192):
    if s not in tf.columns:
        continue
    for name in order:
        t = tf.loc[name, s]
        summary.append({"size": s, "kernel": name, "TFLOPs/s": round(t, 1),
                        "% cuBLAS TF32": round(100 * t / ref_tf32[s], 1),
                        "x cuBLAS FP32": round(t / ref_fp32[s], 2),
                        "% TF32 peak": round(100 * t / PEAK_TF32, 1)})
display(pd.DataFrame(summary).set_index(["size", "kernel"]))

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"1 WMMA naive": "#2a78d6", "2 WMMA+SMEM": "#eb6834", "3 mma.sync": "#1baf7a",
          "4 cp.async": "#eda100", "5 swizzle": "#e87ba4", "6 vec frag": "#8a5cd1", "7 frag pipe": "#008300",
          "cuBLAS TF32": "#52514e", "cuBLAS FP32": "#9a9893"}
sizes = sorted(df["size"].unique())

fig, ax = plt.subplots(figsize=(8, 4.8))
for name in order:
    d = df[df.kernel == name].sort_values("size")
    ax.plot(d["size"], d.tflops, marker="o", markersize=5, linewidth=2, color=COLORS[name],
            linestyle="--" if name.startswith("cuBLAS") else "-", label=name)
ax.axhline(PEAK_TF32, color="#9a9893", linewidth=1, linestyle=":")
ax.text(sizes[0], PEAK_TF32, " TF32 peak 156", va="bottom", fontsize=9, color="#52514e")
ax.set_xscale("log", base=2)
ax.set_xticks(sizes, [str(s) for s in sizes])
ax.set_xlabel("M = N = K")
ax.set_ylabel("TFLOPs/s")
ax.set_title("TF32 Tensor Core GEMM on A100: kernels 1-7 vs cuBLAS")
ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, ncol=4, loc="upper left", bbox_to_anchor=(0, -0.15))
fig.tight_layout()
plt.show()

## 4. SASS 확인

정적 명령 개수로 각 커널이 어떤 명령으로 내려갔는지 확인한다.
`HMMA`는 Tensor Core 명령, `LDGSTS`는 `cp.async`, `FFMA`는 CUDA core FP32 FMA다.

In [ ]:
sass = subprocess.run(["cuobjdump", "-sass", str(BIN)], capture_output=True, text=True).stdout
funcs = re.split(r"\n\s*Function : ", sass)[1:]
rows = []
for f in funcs:
    name = f.split("\n", 1)[0].strip()
    demangled = subprocess.run(["c++filt", name], capture_output=True, text=True).stdout.strip()
    if "tcgemm" not in demangled:
        continue
    rows.append({"kernel": demangled.split("(")[0].replace("void ", ""),
                 **{op: len(re.findall(rf"\b{op}\b", f)) for op in ("HMMA", "LDGSTS", "LDS", "FFMA", "BAR")}})
display(pd.DataFrame(rows).set_index("kernel"))

## 5. 결과 저장

In [ ]:
import json, datetime
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
result = {"date": datetime.date.today().isoformat(), "gpu": gpu,
          "rows": df.to_dict(orient="records"), "logs": logs}
Path("tensor_core_gemm_a100_results.json").write_text(json.dumps(result, indent=1))
print("saved tensor_core_gemm_a100_results.json")